In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import re

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configuration
DATA_DIR = Path('data')
PROCESSED_DIR = Path('..') / DATA_DIR / 'processed'
RESULTS_BASE_DIR = Path('..') / DATA_DIR / 'results'
MODELS_BASE_DIR = Path('..') / DATA_DIR / 'models'

# ===== CONFIGURATION: Set timestamp or leave as None to use latest =====
RESULTS_TIMESTAMP = None # "20251230_042638" # "20251229_1725111"  # e.g., "20251229_1725111" or None for latest
MODELS_TIMESTAMP = None   # e.g., "20251229_1725111" or None for latest
MODEL_NAME = "LightGBM"   # Model name used in filenames
# ======================================================================

def find_latest_timestamped_dir(base_dir):
    """Find the most recent timestamped directory in base_dir."""
    if not base_dir.exists():
        raise FileNotFoundError(f"Directory not found: {base_dir}")
    
    # Pattern: YYYYMMDD_HHMMSS
    timestamp_pattern = re.compile(r"^\d{8}_\d{6}$")
    
    timestamped_dirs = [
        d for d in base_dir.iterdir() 
        if d.is_dir() and timestamp_pattern.match(d.name)
    ]
    
    if not timestamped_dirs:
        raise FileNotFoundError(f"No timestamped directories found in: {base_dir}")
    
    # Sort by directory name (timestamp) descending and return the most recent
    timestamped_dirs.sort(key=lambda d: d.name, reverse=True)
    return timestamped_dirs[0]

def get_timestamped_dir(base_dir, timestamp=None):
    """Get specific timestamped directory or latest if timestamp is None."""
    if timestamp:
        timestamp_dir = base_dir / timestamp
        if not timestamp_dir.exists():
            raise FileNotFoundError(f"Timestamped directory not found: {timestamp_dir}")
        return timestamp_dir
    else:
        return find_latest_timestamped_dir(base_dir)

# Get timestamped directories
try:
    results_dir = get_timestamped_dir(RESULTS_BASE_DIR, RESULTS_TIMESTAMP)
    print(f"Using results from: {results_dir.name}")
except FileNotFoundError as e:
    print(f"Warning: {e}")
    results_dir = None

try:
    models_dir = get_timestamped_dir(MODELS_BASE_DIR, MODELS_TIMESTAMP)
    print(f"Using models from: {models_dir.name}")
except FileNotFoundError as e:
    print(f"Warning: {e}")
    models_dir = None

# File paths for processed data
PANEL_1_PROCESSED = PROCESSED_DIR / "panel_1.parquet"
PANEL_2_PROCESSED = PROCESSED_DIR / "panel_2.parquet"

# File paths for predictions (in timestamped directory)
if results_dir:
    PANEL_1_FLAGS = results_dir / f"{MODEL_NAME}_panel_1_flags.csv"
    PANEL_1_FORECAST = results_dir / f"{MODEL_NAME}_panel_1_forecast.csv"
    PANEL_1_SCORES = results_dir / f"{MODEL_NAME}_panel_1_scores.csv"
    
    PANEL_2_FLAGS = results_dir / f"{MODEL_NAME}_panel_2_flags.csv"
    PANEL_2_FORECAST = results_dir / f"{MODEL_NAME}_panel_2_forecast.csv"
    PANEL_2_SCORES = results_dir / f"{MODEL_NAME}_panel_2_scores.csv"
    
    print(f"\nPrediction files configured successfully!")
else:
    print("\nNo results directory found - please run predictions first!")
    PANEL_1_FLAGS = PANEL_1_FORECAST = PANEL_1_SCORES = None
    PANEL_2_FLAGS = PANEL_2_FORECAST = PANEL_2_SCORES = None


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set plotly template for better visuals
pio.templates.default = "plotly_white"


In [ ]:
# Load processed data
print("Loading processed data...")
df_panel_1 = pd.read_parquet(PANEL_1_PROCESSED)
df_panel_2 = pd.read_parquet(PANEL_2_PROCESSED)

# Ensure timestamp is a column if it's the index
if df_panel_1.index.name == 'timestamp' or isinstance(df_panel_1.index, pd.DatetimeIndex):
    df_panel_1 = df_panel_1.reset_index()
    if 'index' in df_panel_1.columns:
        df_panel_1 = df_panel_1.rename(columns={'index': 'timestamp'})

if df_panel_2.index.name == 'timestamp' or isinstance(df_panel_2.index, pd.DatetimeIndex):
    df_panel_2 = df_panel_2.reset_index()
    if 'index' in df_panel_2.columns:
        df_panel_2 = df_panel_2.rename(columns={'index': 'timestamp'})

# Convert timestamp to datetime if needed
if 'timestamp' in df_panel_1.columns:
    df_panel_1['timestamp'] = pd.to_datetime(df_panel_1['timestamp'])
if 'timestamp' in df_panel_2.columns:
    df_panel_2['timestamp'] = pd.to_datetime(df_panel_2['timestamp'])

print(f"Panel 1: {len(df_panel_1)} rows, columns: {list(df_panel_1.columns)}")
print(f"Panel 2: {len(df_panel_2)} rows, columns: {list(df_panel_2.columns)}")
print("\nFirst few rows of Panel 1:")
print(df_panel_1.head())


In [ ]:
# Load prediction files
print("Loading prediction files...")

if not results_dir or not PANEL_1_FLAGS:
    raise FileNotFoundError("No prediction files configured. Please ensure predictions have been run.")

# Check if files exist
for file_path, name in [
    (PANEL_1_FLAGS, "Panel 1 flags"),
    (PANEL_1_FORECAST, "Panel 1 forecast"),
    (PANEL_1_SCORES, "Panel 1 scores"),
    (PANEL_2_FLAGS, "Panel 2 flags"),
    (PANEL_2_FORECAST, "Panel 2 forecast"),
    (PANEL_2_SCORES, "Panel 2 scores")
]:
    if not file_path.exists():
        raise FileNotFoundError(f"{name} file not found: {file_path}")

# Panel 1 predictions
df_p1_flags = pd.read_csv(PANEL_1_FLAGS)
df_p1_forecast = pd.read_csv(PANEL_1_FORECAST)
df_p1_scores = pd.read_csv(PANEL_1_SCORES)

# Panel 2 predictions
df_p2_flags = pd.read_csv(PANEL_2_FLAGS)
df_p2_forecast = pd.read_csv(PANEL_2_FORECAST)
df_p2_scores = pd.read_csv(PANEL_2_SCORES)

# Convert timestamps
for df in [df_p1_flags, df_p1_forecast, df_p1_scores, df_p2_flags, df_p2_forecast, df_p2_scores]:
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Panel 1 predictions:")
print(f"  Flags: {len(df_p1_flags)} rows")
print(f"  Forecast: {len(df_p1_forecast)} rows, columns: {list(df_p1_forecast.columns)}")
print(f"  Scores: {len(df_p1_scores)} rows")

print("\nPanel 2 predictions:")
print(f"  Flags: {len(df_p2_flags)} rows")
print(f"  Forecast: {len(df_p2_forecast)} rows, columns: {list(df_p2_forecast.columns)}")
print(f"  Scores: {len(df_p2_scores)} rows")


## Merge Data


In [ ]:
# Merge Panel 1 data
df_p1_merged = df_panel_1.copy()
df_p1_merged = df_p1_merged.merge(df_p1_flags, on='timestamp', how='left', suffixes=('', '_flag'))
df_p1_merged = df_p1_merged.merge(df_p1_scores, on='timestamp', how='left', suffixes=('', '_score'))
df_p1_merged = df_p1_merged.merge(df_p1_forecast, on='timestamp', how='left', suffixes=('_actual', '_forecast'))

# Merge Panel 2 data
df_p2_merged = df_panel_2.copy()
df_p2_merged = df_p2_merged.merge(df_p2_flags, on='timestamp', how='left', suffixes=('', '_flag'))
df_p2_merged = df_p2_merged.merge(df_p2_scores, on='timestamp', how='left', suffixes=('', '_score'))
df_p2_merged = df_p2_merged.merge(df_p2_forecast, on='timestamp', how='left', suffixes=('_actual', '_forecast'))

# Rename anomaly columns to standard names (CSV files use '0' and '0_score' as column names)
# Check if the columns exist and rename them
if '0' in df_p1_merged.columns:
    df_p1_merged = df_p1_merged.rename(columns={'0': 'anomaly_flag'})
if '0_score' in df_p1_merged.columns:
    df_p1_merged = df_p1_merged.rename(columns={'0_score': 'anomaly_score'})

if '0' in df_p2_merged.columns:
    df_p2_merged = df_p2_merged.rename(columns={'0': 'anomaly_flag'})
if '0_score' in df_p2_merged.columns:
    df_p2_merged = df_p2_merged.rename(columns={'0_score': 'anomaly_score'})

print("Panel 1 merged shape:", df_p1_merged.shape)
print("Panel 2 merged shape:", df_p2_merged.shape)
print("\nPanel 1 merged columns:", list(df_p1_merged.columns))
print("\nPanel 1 anomaly flag summary:")
if 'anomaly_flag' in df_p1_merged.columns:
    print(f"  Anomalies detected: {df_p1_merged['anomaly_flag'].sum()} (out of {df_p1_merged['anomaly_flag'].notna().sum()} predictions)")


## Visualization Functions


In [ ]:
# Interactive Plotly versions of plotting functions

def plot_channel_with_predictions_interactive(df, panel_id, channel_col, forecast_col=None, title_suffix=""):
    """Plot actual vs forecasted values for a channel with anomaly flags (Interactive Plotly version)."""
    
    # Get anomaly flags and scores
    has_anomaly = df['anomaly_flag'] == 1 if 'anomaly_flag' in df.columns else pd.Series([False] * len(df))
    anomaly_scores = df['anomaly_score'] if 'anomaly_score' in df.columns else None
    
    # Get time range for zooming
    time_range = get_prediction_time_range(df)
    
    # Clean up column name for title
    clean_col_name = channel_col.replace('_actual', '') if channel_col.endswith('_actual') else channel_col
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=(f'Panel {panel_id} - {clean_col_name}{title_suffix}', 'Anomaly Scores'),
        vertical_spacing=0.12,
        row_heights=[0.6, 0.4]
    )
    
    # Plot 1: Actual vs Forecast
    # Actual values
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'], 
            y=df[channel_col],
            mode='lines',
            name='Actual',
            line=dict(color='#1f77b4', width=2),
            hovertemplate='<b>Actual</b><br>Time: %{x}<br>Value: %{y:.4f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Forecast values
    if forecast_col and forecast_col in df.columns:
        fig.add_trace(
            go.Scatter(
                x=df['timestamp'], 
                y=df[forecast_col],
                mode='lines',
                name='Forecast',
                line=dict(color='#ff7f0e', width=2, dash='dash'),
                hovertemplate='<b>Forecast</b><br>Time: %{x}<br>Value: %{y:.4f}<extra></extra>'
            ),
            row=1, col=1
        )
    
    # Highlight anomalies
    if has_anomaly.any():
        anomaly_times = df.loc[has_anomaly, 'timestamp']
        anomaly_values = df.loc[has_anomaly, channel_col]
        fig.add_trace(
            go.Scatter(
                x=anomaly_times, 
                y=anomaly_values,
                mode='markers',
                name='Anomaly',
                marker=dict(color='red', size=8, symbol='circle', opacity=0.7),
                hovertemplate='<b>Anomaly</b><br>Time: %{x}<br>Value: %{y:.4f}<extra></extra>'
            ),
            row=1, col=1
        )
    
    # Plot 2: Anomaly Scores
    if anomaly_scores is not None:
        fig.add_trace(
            go.Scatter(
                x=df['timestamp'], 
                y=anomaly_scores,
                mode='lines',
                name='Anomaly Score',
                fill='tozeroy',
                line=dict(color='purple', width=2),
                fillcolor='rgba(128, 0, 128, 0.3)',
                hovertemplate='<b>Anomaly Score</b><br>Time: %{x}<br>Score: %{y:.2f}<extra></extra>'
            ),
            row=2, col=1
        )
        
        # Add threshold line if we can infer it
        if has_anomaly.any():
            threshold_estimate = anomaly_scores[has_anomaly].min()
            if threshold_estimate:
                fig.add_hline(
                    y=threshold_estimate, 
                    line_dash="dash", 
                    line_color="red",
                    opacity=0.5,
                    annotation_text="Estimated Threshold",
                    row=2, col=1
                )
    
    # Update axes labels
    fig.update_xaxes(title_text="Timestamp", row=2, col=1)
    fig.update_yaxes(title_text="Value", row=1, col=1)
    fig.update_yaxes(title_text="Anomaly Score", row=2, col=1)
    
    # Set zoom range if available
    if time_range:
        fig.update_xaxes(range=[time_range[0], time_range[1]], row=1, col=1)
        fig.update_xaxes(range=[time_range[0], time_range[1]], row=2, col=1)
    
    # Update layout
    fig.update_layout(
        height=800,
        showlegend=True,
        hovermode='x unified',
        template='plotly_white'
    )
    
    return fig

def get_prediction_time_range(df, padding_hours=2):
    """Get the time range where predictions/anomalies exist, with padding.
    
    Looks for rows where anomaly_flag or anomaly_score is not null (including 0/False values).
    """
    # Find where we have predictions - check for any non-null values in prediction columns
    prediction_mask = pd.Series([False] * len(df), index=df.index)
    
    if 'anomaly_flag' in df.columns:
        # Include both True (1) and False (0) - any non-null means prediction exists
        prediction_mask = prediction_mask | df['anomaly_flag'].notna()
    
    if 'anomaly_score' in df.columns:
        prediction_mask = prediction_mask | df['anomaly_score'].notna()
    
    if prediction_mask.any():
        prediction_times = df.loc[prediction_mask, 'timestamp']
        if len(prediction_times) > 0:
            time_min = prediction_times.min()
            time_max = prediction_times.max()
            # Add padding
            padding = pd.Timedelta(hours=padding_hours)
            return time_min - padding, time_max + padding
    
    # If no predictions found, return None to use full range
    return None

def plot_all_channels_interactive(df, panel_id, title_suffix=""):
    """Plot all channels in subplots (Interactive Plotly version)."""
    # Get channel columns (exclude timestamp, anomaly_flag, anomaly_score)
    exclude_cols = ['timestamp', 'anomaly_flag', 'anomaly_score']
    channel_cols = [col for col in df.columns if col not in exclude_cols and not col.endswith('_forecast')]
    
    # Get actual columns - they may have _actual suffix or not
    actual_cols = [col for col in channel_cols if col.endswith('_actual')]
    
    # If no _actual columns found, try without suffix
    if not actual_cols:
        actual_cols = [col for col in channel_cols if not col.endswith('_forecast')]
    
    n_channels = len(actual_cols)
    n_cols = 2
    n_rows = (n_channels + 1) // 2
    
    has_anomaly = df['anomaly_flag'] == 1 if 'anomaly_flag' in df.columns else pd.Series([False] * len(df))
    
    # Get time range for zooming (apply to all subplots)
    time_range = get_prediction_time_range(df)
    
    # Create subplot titles
    subplot_titles = [col.replace('_actual', '') if col.endswith('_actual') else col 
                      for col in actual_cols]
    
    # Create subplots
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles,
        vertical_spacing=0.08,
        horizontal_spacing=0.08
    )
    
    for idx, col in enumerate(actual_cols):
        row = idx // n_cols + 1
        col_num = idx % n_cols + 1
        
        # Plot actual
        fig.add_trace(
            go.Scatter(
                x=df['timestamp'], 
                y=df[col],
                mode='lines',
                name='Actual',
                line=dict(color='#1f77b4', width=1.5),
                showlegend=(idx == 0),  # Only show legend for first trace
                hovertemplate='<b>Actual</b><br>Time: %{x}<br>Value: %{y:.4f}<extra></extra>',
                legendgroup='actual'
            ),
            row=row, col=col_num
        )
        
        # Plot forecast if available - construct forecast column name
        if col.endswith('_actual'):
            forecast_col = col.replace('_actual', '_forecast')
        else:
            forecast_col = col + '_forecast'
        
        if forecast_col in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df['timestamp'], 
                    y=df[forecast_col],
                    mode='lines',
                    name='Forecast',
                    line=dict(color='#ff7f0e', width=1.5, dash='dash'),
                    showlegend=(idx == 0),
                    hovertemplate='<b>Forecast</b><br>Time: %{x}<br>Value: %{y:.4f}<extra></extra>',
                    legendgroup='forecast'
                ),
                row=row, col=col_num
            )
        
        # Highlight anomalies
        if has_anomaly.any():
            anomaly_times = df.loc[has_anomaly, 'timestamp']
            anomaly_values = df.loc[has_anomaly, col]
            fig.add_trace(
                go.Scatter(
                    x=anomaly_times, 
                    y=anomaly_values,
                    mode='markers',
                    name='Anomaly',
                    marker=dict(color='red', size=5, symbol='circle', opacity=0.7),
                    showlegend=(idx == 0),
                    hovertemplate='<b>Anomaly</b><br>Time: %{x}<br>Value: %{y:.4f}<extra></extra>',
                    legendgroup='anomaly'
                ),
                row=row, col=col_num
            )
        
        # Set zoom range if available
        if time_range:
            fig.update_xaxes(range=[time_range[0], time_range[1]], row=row, col=col_num)
        
        # Update y-axis label
        fig.update_yaxes(title_text="Value", row=row, col=col_num)
    
    # Update layout
    fig.update_layout(
        height=400 * n_rows,
        title_text=f'Panel {panel_id} - All Channels{title_suffix}',
        title_font_size=16,
        showlegend=True,
        hovermode='closest',
        template='plotly_white'
    )
    
    return fig

print("Interactive Plotly visualization functions defined!")


In [ ]:
def plot_channel_with_predictions(df, panel_id, channel_col, forecast_col=None, title_suffix=""):
    """Plot actual vs forecasted values for a channel with anomaly flags."""
    fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    
    # Get anomaly flags and scores
    has_anomaly = df['anomaly_flag'] == 1 if 'anomaly_flag' in df.columns else pd.Series([False] * len(df))
    anomaly_scores = df['anomaly_score'] if 'anomaly_score' in df.columns else None
    
    # Get time range for zooming
    time_range = get_prediction_time_range(df)
    
    # Plot 1: Actual vs Forecast
    ax1 = axes[0]
    ax1.plot(df['timestamp'], df[channel_col], label='Actual', alpha=0.7, linewidth=1.5)
    
    if forecast_col and forecast_col in df.columns:
        ax1.plot(df['timestamp'], df[forecast_col], label='Forecast', alpha=0.7, 
                linestyle='--', linewidth=1.5, color='orange')
    
    # Highlight anomalies
    if has_anomaly.any():
        anomaly_times = df.loc[has_anomaly, 'timestamp']
        anomaly_values = df.loc[has_anomaly, channel_col]
        ax1.scatter(anomaly_times, anomaly_values, color='red', s=50, 
                   alpha=0.6, label='Anomaly', zorder=5)
    
    # Zoom to prediction time range
    if time_range:
        ax1.set_xlim(time_range[0], time_range[1])
    
    ax1.set_ylabel('Value', fontsize=12)
    # Clean up column name for title
    clean_col_name = channel_col.replace('_actual', '') if channel_col.endswith('_actual') else channel_col
    ax1.set_title(f'Panel {panel_id} - {clean_col_name}{title_suffix}', fontsize=14, fontweight='bold')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Anomaly Scores
    ax2 = axes[1]
    if anomaly_scores is not None:
        ax2.plot(df['timestamp'], anomaly_scores, label='Anomaly Score', 
                color='purple', linewidth=1.5)
        ax2.fill_between(df['timestamp'], anomaly_scores, alpha=0.3, color='purple')
    
    # Add threshold line if we can infer it
    if anomaly_scores is not None and has_anomaly.any():
        # Show where anomalies were flagged
        threshold_estimate = anomaly_scores[has_anomaly].min() if has_anomaly.any() else None
        if threshold_estimate:
            ax2.axhline(y=threshold_estimate, color='red', linestyle='--', 
                       alpha=0.5, label='Estimated Threshold')
    
    # Zoom to prediction time range
    if time_range:
        ax2.set_xlim(time_range[0], time_range[1])
    
    ax2.set_ylabel('Anomaly Score', fontsize=12)
    ax2.set_xlabel('Timestamp', fontsize=12)
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

def plot_all_channels(df, panel_id, title_suffix=""):
    """Plot all channels in subplots."""
    # Get channel columns (exclude timestamp, anomaly_flag, anomaly_score)
    exclude_cols = ['timestamp', 'anomaly_flag', 'anomaly_score']
    channel_cols = [col for col in df.columns if col not in exclude_cols and not col.endswith('_forecast')]
    
    # Get actual columns - they may have _actual suffix or not
    actual_cols = [col for col in channel_cols if col.endswith('_actual')]
    
    # If no _actual columns found, try without suffix
    if not actual_cols:
        actual_cols = [col for col in channel_cols if not col.endswith('_forecast')]
    
    n_channels = len(actual_cols)
    n_cols = 2
    n_rows = (n_channels + 1) // 2
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
    axes = axes.flatten() if n_channels > 1 else [axes]
    
    has_anomaly = df['anomaly_flag'] == 1 if 'anomaly_flag' in df.columns else pd.Series([False] * len(df))
    
    # Get time range for zooming (apply to all subplots)
    time_range = get_prediction_time_range(df)
    
    for idx, col in enumerate(actual_cols):
        ax = axes[idx]
        
        # Plot actual
        ax.plot(df['timestamp'], df[col], label='Actual', alpha=0.7, linewidth=1.5)
        
        # Plot forecast if available - construct forecast column name
        if col.endswith('_actual'):
            forecast_col = col.replace('_actual', '_forecast')
        else:
            forecast_col = col + '_forecast'
        
        if forecast_col in df.columns:
            ax.plot(df['timestamp'], df[forecast_col], label='Forecast', 
                   alpha=0.7, linestyle='--', linewidth=1.5, color='orange')
        
        # Highlight anomalies
        if has_anomaly.any():
            anomaly_times = df.loc[has_anomaly, 'timestamp']
            anomaly_values = df.loc[has_anomaly, col]
            ax.scatter(anomaly_times, anomaly_values, color='red', s=30, 
                      alpha=0.6, label='Anomaly', zorder=5)
        
        # Zoom to prediction time range
        if time_range:
            ax.set_xlim(time_range[0], time_range[1])
        
        # Clean up column name for title (remove _actual suffix)
        title = col.replace('_actual', '') if col.endswith('_actual') else col
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_ylabel('Value', fontsize=10)
        ax.legend(loc='best', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(len(actual_cols), len(axes)):
        axes[idx].set_visible(False)
    
    fig.suptitle(f'Panel {panel_id} - All Channels{title_suffix}', 
                fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    return fig

print("Visualization functions defined!")


## Panel 1 Visualizations


In [ ]:
# Summary statistics for Panel 1
print("Panel 1 Summary:")
print(f"Total data points: {len(df_p1_merged)}")
if 'anomaly_flag' in df_p1_merged.columns:
    anomaly_count = df_p1_merged['anomaly_flag'].sum()
    anomaly_pct = (anomaly_count / len(df_p1_merged)) * 100
    print(f"Anomalies detected: {anomaly_count} ({anomaly_pct:.2f}%)")
if 'anomaly_score' in df_p1_merged.columns:
    print(f"Anomaly score range: {df_p1_merged['anomaly_score'].min():.2f} - {df_p1_merged['anomaly_score'].max():.2f}")
    print(f"Anomaly score mean: {df_p1_merged['anomaly_score'].mean():.2f}")

print("\nDate range:", df_p1_merged['timestamp'].min(), "to", df_p1_merged['timestamp'].max())


In [ ]:
# Plot all channels (Interactive) for Panel 1
fig = plot_all_channels_interactive(df_p1_merged, panel_id=1, title_suffix=" - All Channels")
fig.show()


In [ ]:
# Plot individual channels with detailed view (Interactive)
# Get actual columns (those ending with _actual)
channel_cols = [col for col in df_p1_merged.columns 
                if col not in ['timestamp', 'anomaly_flag', 'anomaly_score'] 
                and col.endswith('_actual')]

# Plot first few channels in detail
for col in channel_cols:  # Plot first 3 channels
    # Construct forecast column name
    forecast_col = col.replace('_actual', '_forecast')
    fig = plot_channel_with_predictions_interactive(df_p1_merged, panel_id=1, 
                                       channel_col=col, forecast_col=forecast_col)
    fig.show()


## Panel 2 Visualizations


In [ ]:
# Summary statistics for Panel 2
print("Panel 2 Summary:")
print(f"Total data points: {len(df_p2_merged)}")
if 'anomaly_flag' in df_p2_merged.columns:
    anomaly_count = df_p2_merged['anomaly_flag'].sum()
    anomaly_pct = (anomaly_count / len(df_p2_merged)) * 100
    print(f"Anomalies detected: {anomaly_count} ({anomaly_pct:.2f}%)")
if 'anomaly_score' in df_p2_merged.columns:
    print(f"Anomaly score range: {df_p2_merged['anomaly_score'].min():.2f} - {df_p2_merged['anomaly_score'].max():.2f}")
    print(f"Anomaly score mean: {df_p2_merged['anomaly_score'].mean():.2f}")

print("\nDate range:", df_p2_merged['timestamp'].min(), "to", df_p2_merged['timestamp'].max())


In [ ]:
# Plot all channels (Interactive) for Panel 2
fig = plot_all_channels_interactive(df_p2_merged, panel_id=2, title_suffix=" - All Channels")
fig.show()


In [ ]:
# Plot individual channels with detailed view (Interactive) for Panel 2
# Get actual columns (those ending with _actual)
channel_cols_p2 = [col for col in df_p2_merged.columns 
                   if col not in ['timestamp', 'anomaly_flag', 'anomaly_score'] 
                   and col.endswith('_actual')]

# Plot first few channels in detail
for col in channel_cols_p2[:3]:  # Plot first 3 channels
    # Construct forecast column name
    forecast_col = col.replace('_actual', '_forecast')
    fig = plot_channel_with_predictions_interactive(df_p2_merged, panel_id=2, 
                                       channel_col=col, forecast_col=forecast_col)
    fig.show()


## Anomaly Analysis


In [ ]:
# Compare anomaly detection between panels
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Get time range for zooming
time_range_p1 = get_prediction_time_range(df_p1_merged)
time_range_p2 = get_prediction_time_range(df_p2_merged)
# Use the union of both ranges if both exist
if time_range_p1 and time_range_p2:
    time_range = (min(time_range_p1[0], time_range_p2[0]), 
                  max(time_range_p1[1], time_range_p2[1]))
elif time_range_p1:
    time_range = time_range_p1
elif time_range_p2:
    time_range = time_range_p2
else:
    time_range = None

# Panel 1 anomaly scores over time
ax1 = axes[0]
ax1.plot(df_p1_merged['timestamp'], df_p1_merged['anomaly_score'], 
        label='Anomaly Score', color='blue', linewidth=1.5)
ax1.fill_between(df_p1_merged['timestamp'], df_p1_merged['anomaly_score'], 
                alpha=0.3, color='blue')
if 'anomaly_flag' in df_p1_merged.columns:
    anomalies_p1 = df_p1_merged[df_p1_merged['anomaly_flag'] == 1]
    if len(anomalies_p1) > 0:
        ax1.scatter(anomalies_p1['timestamp'], anomalies_p1['anomaly_score'], 
                   color='red', s=50, alpha=0.7, label='Flagged Anomalies', zorder=5)

# Zoom to prediction time range
if time_range:
    ax1.set_xlim(time_range[0], time_range[1])

ax1.set_ylabel('Anomaly Score', fontsize=12)
ax1.set_title('Panel 1 - Anomaly Scores Over Time', fontsize=14, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Panel 2 anomaly scores over time
ax2 = axes[1]
ax2.plot(df_p2_merged['timestamp'], df_p2_merged['anomaly_score'], 
        label='Anomaly Score', color='green', linewidth=1.5)
ax2.fill_between(df_p2_merged['timestamp'], df_p2_merged['anomaly_score'], 
                alpha=0.3, color='green')
if 'anomaly_flag' in df_p2_merged.columns:
    anomalies_p2 = df_p2_merged[df_p2_merged['anomaly_flag'] == 1]
    if len(anomalies_p2) > 0:
        ax2.scatter(anomalies_p2['timestamp'], anomalies_p2['anomaly_score'], 
                   color='red', s=50, alpha=0.7, label='Flagged Anomalies', zorder=5)

# Zoom to prediction time range
if time_range:
    ax2.set_xlim(time_range[0], time_range[1])

ax2.set_ylabel('Anomaly Score', fontsize=12)
ax2.set_xlabel('Timestamp', fontsize=12)
ax2.set_title('Panel 2 - Anomaly Scores Over Time', fontsize=14, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Anomaly statistics comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly count comparison
ax1 = axes[0]
panels = ['Panel 1', 'Panel 2']
anomaly_counts = [
    df_p1_merged['anomaly_flag'].sum() if 'anomaly_flag' in df_p1_merged.columns else 0,
    df_p2_merged['anomaly_flag'].sum() if 'anomaly_flag' in df_p2_merged.columns else 0
]
bars = ax1.bar(panels, anomaly_counts, color=['blue', 'green'], alpha=0.7)
ax1.set_ylabel('Number of Anomalies', fontsize=12)
ax1.set_title('Total Anomalies Detected', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
for bar, count in zip(bars, anomaly_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(anomaly_counts)*0.01,
            f'{count}', ha='center', va='bottom', fontweight='bold')

# Anomaly score distribution
ax2 = axes[1]
if 'anomaly_score' in df_p1_merged.columns and 'anomaly_score' in df_p2_merged.columns:
    ax2.hist(df_p1_merged['anomaly_score'].dropna(), bins=50, alpha=0.6, 
            label='Panel 1', color='blue', density=True)
    ax2.hist(df_p2_merged['anomaly_score'].dropna(), bins=50, alpha=0.6, 
            label='Panel 2', color='green', density=True)
    ax2.set_xlabel('Anomaly Score', fontsize=12)
    ax2.set_ylabel('Density', fontsize=12)
    ax2.set_title('Anomaly Score Distribution', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Forecast Accuracy Analysis


In [ ]:
# Calculate forecast errors for Panel 1
def calculate_forecast_metrics(df, panel_id):
    """Calculate forecast accuracy metrics."""
    metrics = {}
    
    # Get actual columns (those ending with _actual)
    actual_cols = [col for col in df.columns 
                   if col not in ['timestamp', 'anomaly_flag', 'anomaly_score'] 
                   and col.endswith('_actual')]
    
    for col in actual_cols:
        # Construct forecast column name
        forecast_col = col.replace('_actual', '_forecast')
        if forecast_col in df.columns:
            # Calculate errors
            actual = df[col].dropna()
            forecast = df[forecast_col].dropna()
            
            # Align indices
            common_idx = actual.index.intersection(forecast.index)
            if len(common_idx) > 0:
                actual_aligned = actual.loc[common_idx]
                forecast_aligned = forecast.loc[common_idx]
                
                # Calculate metrics
                mae = np.mean(np.abs(actual_aligned - forecast_aligned))
                rmse = np.sqrt(np.mean((actual_aligned - forecast_aligned)**2))
                # Avoid division by zero in MAPE
                mape = np.mean(np.abs((actual_aligned - forecast_aligned) / (actual_aligned + 1e-10))) * 100
                
                # Use clean column name (without _actual suffix) for metrics dict
                clean_col_name = col.replace('_actual', '')
                metrics[clean_col_name] = {
                    'MAE': mae,
                    'RMSE': rmse,
                    'MAPE': mape,
                    'n_points': len(common_idx)
                }
    
    return metrics

metrics_p1 = calculate_forecast_metrics(df_p1_merged, 1)
metrics_p2 = calculate_forecast_metrics(df_p2_merged, 2)

print("Panel 1 Forecast Metrics:")
for channel, metric in metrics_p1.items():
    print(f"\n{channel}:")
    print(f"  MAE: {metric['MAE']:.4f}")
    print(f"  RMSE: {metric['RMSE']:.4f}")
    print(f"  MAPE: {metric['MAPE']:.2f}%")
    print(f"  Data points: {metric['n_points']}")

print("\n" + "="*50)
print("Panel 2 Forecast Metrics:")
for channel, metric in metrics_p2.items():
    print(f"\n{channel}:")
    print(f"  MAE: {metric['MAE']:.4f}")
    print(f"  RMSE: {metric['RMSE']:.4f}")
    print(f"  MAPE: {metric['MAPE']:.2f}%")
    print(f"  Data points: {metric['n_points']}")


In [ ]:
# Visualize forecast errors
if metrics_p1 or metrics_p2:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Panel 1 RMSE
    if metrics_p1:
        ax1 = axes[0]
        channels = list(metrics_p1.keys())
        rmse_values = [metrics_p1[ch]['RMSE'] for ch in channels]
        bars = ax1.barh(channels, rmse_values, color='blue', alpha=0.7)
        ax1.set_xlabel('RMSE', fontsize=12)
        ax1.set_title('Panel 1 - Forecast RMSE by Channel', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3, axis='x')
        for bar, val in zip(bars, rmse_values):
            ax1.text(val + max(rmse_values)*0.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center', fontsize=9)
    
    # Panel 2 RMSE
    if metrics_p2:
        ax2 = axes[1]
        channels = list(metrics_p2.keys())
        rmse_values = [metrics_p2[ch]['RMSE'] for ch in channels]
        bars = ax2.barh(channels, rmse_values, color='green', alpha=0.7)
        ax2.set_xlabel('RMSE', fontsize=12)
        ax2.set_title('Panel 2 - Forecast RMSE by Channel', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='x')
        for bar, val in zip(bars, rmse_values):
            ax2.text(val + max(rmse_values)*0.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()


## Data Export
